# DoDAF Minimal Second Pack Proof

This notebook is the Phase 7 deep dive for the local `dodaf_minimal` second-pack slice.

It proves three things:

1. one pack can support both a strict and a mixed profile;
2. the same review and overlay workflow works for the second pack;
3. the existing CLI surface can drive that workflow without pack-specific branching.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

SEARCH_ROOTS = [
    Path.cwd().resolve(),
    Path('~/projects/onto-canon6').expanduser(),
]

PROJECT_ROOT = None
for start in SEARCH_ROOTS:
    candidate = start
    while True:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            PROJECT_ROOT = candidate
            break
        if candidate.parent == candidate:
            break
        candidate = candidate.parent
    if PROJECT_ROOT is not None:
        break

if PROJECT_ROOT is None:
    raise RuntimeError(
        "Could not locate onto-canon6 repo root from notebook cwd or ~/projects/onto-canon6"
    )

for candidate_path in (
    PROJECT_ROOT / "src",
    PROJECT_ROOT.parent / "llm_client",
    Path('~/projects/llm_client').expanduser(),
):
    if candidate_path.exists():
        candidate_str = str(candidate_path)
        if candidate_str not in sys.path:
            sys.path.insert(0, candidate_str)

import contextlib
import io
import json
from pprint import pprint
from tempfile import TemporaryDirectory

from onto_canon6 import cli as cli_module
from onto_canon6.ontology_runtime import load_effective_profile, load_profile, validate_assertion_payload
from onto_canon6.pipeline import (
    CandidateAssertionImport,
    EvidenceSpan,
    ProfileRef,
    ReviewService,
    SourceArtifactRef,
)


## Strict and Mixed Profiles Over The Same Pack

This section proves the model you called out explicitly: the ontology pack stays the same, while strict vs mixed is a profile/policy distinction layered over that pack.

In [2]:
strict_profile = load_profile('dodaf_minimal_strict', '0.1.0')
mixed_profile = load_profile('dodaf_minimal_mixed', '0.1.0')
pprint({
    'strict_mode': strict_profile.ontology_policy.mode,
    'mixed_mode': mixed_profile.ontology_policy.mode,
    'shared_pack_ref': strict_profile.pack_ref.model_dump() if strict_profile.pack_ref is not None else None,
    'mixed_overlay_target': mixed_profile.ontology_policy.overlay_target.model_dump() if mixed_profile.ontology_policy.overlay_target is not None else None,
})


{'mixed_mode': 'mixed',
 'mixed_overlay_target': {'pack_id': 'dodaf_minimal__overlay',
                          'pack_version': '0.1.0'},
 'shared_pack_ref': {'pack_id': 'dodaf_minimal', 'pack_version': '0.1.0'},
 'strict_mode': 'closed'}


## Strict Validation Proof

The strict profile should accept a known predicate from the pack and fail loudly on unknown predicates.

In [3]:
known_outcome = validate_assertion_payload(
    {
        'predicate': 'dodaf:operational_node_exchanges_information',
        'roles': {
            'source_node': [{'kind': 'entity', 'entity_id': 'ent:node:source', 'entity_type': 'dm2:OperationalNode'}],
            'target_node': [{'kind': 'entity', 'entity_id': 'ent:node:target', 'entity_type': 'dm2:OperationalNode'}],
            'information_element': [{'kind': 'entity', 'entity_id': 'ent:info:message', 'entity_type': 'dm2:InformationElement'}],
        },
    },
    profile=strict_profile,
)
unknown_outcome = validate_assertion_payload(
    {
        'predicate': 'dodaf:operational_node_supports_activity',
        'roles': {
            'source': [{'kind': 'value', 'value_kind': 'string', 'value': 'Node A'}],
            'target': [{'kind': 'value', 'value_kind': 'string', 'value': 'Activity B'}],
        },
    },
    profile=strict_profile,
)
{
    'known_hard_errors': [finding.code for finding in known_outcome.hard_errors],
    'unknown_hard_errors': [finding.code for finding in unknown_outcome.hard_errors],
}


{'known_hard_errors': [],
 'unknown_hard_errors': ['oc:profile_unknown_predicate']}

## Mixed Review And Overlay Proof

The mixed profile should route the unknown predicate into the same review and overlay workflow.

In [4]:
with TemporaryDirectory() as tmp_dir:
    tmp_path = Path(tmp_dir)
    review_service = ReviewService(
        db_path=tmp_path / 'review.sqlite3',
        overlay_root=tmp_path / 'overlays',
    )
    submission = review_service.submit_candidate_assertion(
        payload={
            'predicate': 'dodaf:operational_node_supports_activity',
            'roles': {
                'source': [{'kind': 'value', 'value_kind': 'string', 'value': 'Node A'}],
                'target': [{'kind': 'value', 'value_kind': 'string', 'value': 'Activity B'}],
            },
        },
        profile_id='dodaf_minimal_mixed',
        profile_version='0.1.0',
        submitted_by='analyst:notebook',
        source_kind='text_file',
        source_ref='dodaf_source.txt',
    )
    proposal = submission.proposals[0]
    review_service.review_proposal(
        proposal_id=proposal.proposal_id,
        decision='accepted',
        actor_id='analyst:reviewer',
        acceptance_policy='apply_to_overlay',
    )
    from onto_canon6.pipeline import OverlayApplicationService
    overlay_service = OverlayApplicationService(
        db_path=tmp_path / 'review.sqlite3',
        overlay_root=tmp_path / 'overlays',
    )
    overlay_record = overlay_service.apply_proposal_to_overlay(
        proposal_id=proposal.proposal_id,
        applied_by='analyst:reviewer',
    )
    effective_profile = load_effective_profile(
        'dodaf_minimal_mixed',
        '0.1.0',
        overlay_root=tmp_path / 'overlays',
    )
    pprint({
        'validation_status': submission.candidate.validation_status,
        'proposal_target_pack': proposal.target_pack.model_dump() if proposal.target_pack is not None else None,
        'overlay_content_path': overlay_record.content_path,
        'active_overlay_predicates': sorted(effective_profile.active_overlay_predicates),
    })


{'active_overlay_predicates': ['dodaf:operational_node_supports_activity'],
 'overlay_content_path': '/tmp/tmpx07kqmzw/overlays/dodaf_minimal__overlay/0.1.0/predicate_additions.jsonl',
 'proposal_target_pack': {'pack_id': 'dodaf_minimal__overlay',
                          'pack_version': '0.1.0'},
 'validation_status': 'needs_review'}


## CLI Proof With A Deterministic Extractor Stand-In

The second-pack proof should also show that the existing CLI can run the mixed-profile workflow without new pack-specific command logic.

In [5]:
class DeterministicDodafExtractionService:
    """Provide a local stand-in for the remote extraction boundary."""

    def __init__(self, *, review_service: ReviewService) -> None:
        self._review_service = review_service

    def extract_and_submit(
        self,
        *,
        source_text: str,
        profile_id: str,
        profile_version: str,
        submitted_by: str,
        source_ref: str,
        source_kind: str = 'text_file',
        source_label: str | None = None,
        source_metadata: dict[str, object] | None = None,
    ) -> tuple[object, ...]:
        del source_metadata
        phrase = 'supports activity'
        start_char = source_text.index(phrase)
        end_char = start_char + len(phrase)
        candidate_import = CandidateAssertionImport(
            profile=ProfileRef(profile_id=profile_id, profile_version=profile_version),
            payload={
                'predicate': 'dodaf:operational_node_supports_activity',
                'roles': {
                    'source': [{'kind': 'value', 'value_kind': 'string', 'value': 'Node A'}],
                    'target': [{'kind': 'value', 'value_kind': 'string', 'value': 'Activity B'}],
                },
            },
            submitted_by=submitted_by,
            source_artifact=SourceArtifactRef(
                source_kind=source_kind,
                source_ref=source_ref,
                source_label=source_label,
                source_metadata={},
                content_text=source_text,
            ),
            evidence_spans=(EvidenceSpan(start_char=start_char, end_char=end_char, text=phrase),),
            claim_text='Node A supports activity B.',
        )
        return (self._review_service.submit_candidate_import(candidate_import=candidate_import),)

def run_cli(args: list[str]) -> tuple[int, str, str]:
    stdout_buffer = io.StringIO()
    stderr_buffer = io.StringIO()
    with contextlib.redirect_stdout(stdout_buffer), contextlib.redirect_stderr(stderr_buffer):
        exit_code = cli_module.main(args)
    return exit_code, stdout_buffer.getvalue(), stderr_buffer.getvalue()

with TemporaryDirectory() as tmp_dir:
    tmp_path = Path(tmp_dir)
    source_path = tmp_path / 'dodaf_source.txt'
    source_path.write_text('Node A supports activity B through the exchange plan.', encoding='utf-8')
    review_db_path = tmp_path / 'review.sqlite3'
    overlay_root = tmp_path / 'ontology_overlays'
    original_extractor = cli_module.TextExtractionService
    cli_module.TextExtractionService = DeterministicDodafExtractionService
    try:
        extract_exit, extract_stdout, extract_stderr = run_cli([
            'extract-text',
            '--input', str(source_path),
            '--profile-id', 'dodaf_minimal_mixed',
            '--profile-version', '0.1.0',
            '--submitted-by', 'analyst:notebook',
            '--review-db-path', str(review_db_path),
            '--overlay-root', str(overlay_root),
            '--output', 'json',
        ])
        assert extract_exit == 0, extract_stderr
        extract_result = json.loads(extract_stdout)
        pprint(extract_result[0])
    finally:
        cli_module.TextExtractionService = original_extractor


{'candidate': {'candidate_id': 'cand_4c249f8b1ea7416386ce30d8',
               'claim_text': 'Node A supports activity B.',
               'evidence_spans': [{'end_char': 24,
                                   'start_char': 7,
                                   'text': 'supports activity'}],
               'normalized_payload': {'predicate': 'dodaf:operational_node_supports_activity',
                                      'roles': {'source': [{'kind': 'value',
                                                            'value': 'Node A',
                                                            'value_kind': 'string'}],
                                                'target': [{'kind': 'value',
                                                            'value': 'Activity '
                                                                     'B',
                                                            'value_kind': 'string'}]}},
               'payload': {'predicate': 'dodaf:ope